# Cowboys Rebuild Analysis
Testing a 4-step plan to fix the Dallas Cowboys with NFL data (nflverse + Over The Cap).

Run top to bottom in Google Colab. Each section saves its charts and CSVs.

In [ ]:
!pip install nflreadpy -q
import nflreadpy as nfl
import pandas as pd
import matplotlib.pyplot as plt
contracts = nfl.load_contracts().to_pandas()

## Take 1: Trade Dak
Salary vs. efficiency (EPA per dropback) this season, and cash paid per playoff win since 2016.

In [ ]:
import matplotlib.pyplot as plt

# This season's QB stats
this_year = nfl.load_player_stats([2026], summary_level="reg").to_pandas()
qb26 = this_year[this_year["position"] == "QB"].copy()
qb26["dropbacks"] = qb26["attempts"] + qb26["sacks_suffered"]
qb26["epa_per_db"] = qb26["passing_epa"] / qb26["dropbacks"]
qb26 = qb26[qb26["dropbacks"] >= 200]

# Each QB's current contract (biggest active deal)
active = contracts[(contracts["is_active"]) & (contracts["position"] == "QB")]
active = active.sort_values("apy", ascending=False).drop_duplicates("gsis_id")

chart = qb26.merge(active[["gsis_id", "apy", "apy_cap_pct"]], left_on="player_id", right_on="gsis_id", how="inner")

# Print the table so we can sanity check salaries
display(chart[["player_display_name", "dropbacks", "epa_per_db", "apy"]].sort_values("apy", ascending=False).round(3))

# Chart
fig, ax = plt.subplots(figsize=(10, 7))
others = chart[chart["player_display_name"] != "Dak Prescott"]
dak = chart[chart["player_display_name"] == "Dak Prescott"]
ax.scatter(others["apy"], others["epa_per_db"], color="#9AA5B1", s=70)
ax.scatter(dak["apy"], dak["epa_per_db"], color="#003594", s=220, edgecolor="black", zorder=3)
for _, r in chart.iterrows():
    ax.annotate(r["player_display_name"].split()[-1], (r["apy"], r["epa_per_db"]),
                xytext=(5, 5), textcoords="offset points", fontsize=9,
                fontweight="bold" if r["player_display_name"] == "Dak Prescott" else "normal")
ax.axhline(chart["epa_per_db"].mean(), color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Salary per year ($ millions)")
ax.set_ylabel("EPA per dropback (2026 season)")
ax.set_title("Paid like a top-2 QB, playing like a top-7 QB", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("take1_salary_vs_epa.png", dpi=200)
plt.show()

In [ ]:
# Every game since 2016, keep playoff games only
games = nfl.load_schedules().to_pandas()
playoffs = games[(games["game_type"] != "REG") & (games["season"] >= 2016) & (games["result"].notna())]

# Count playoff wins and losses for each starting QB
rows = []
for _, g in playoffs.iterrows():
    home_won = g["result"] > 0
    rows.append({"qb_id": g["home_qb_id"], "wins": int(home_won), "losses": int(not home_won)})
    rows.append({"qb_id": g["away_qb_id"], "wins": int(not home_won), "losses": int(home_won)})
playoff_record = pd.DataFrame(rows).groupby("qb_id").sum().reset_index()

# Career regular season efficiency since 2016
career = nfl.load_player_stats(list(range(2016, 2027)), summary_level="reg").to_pandas()
career = career[career["position"] == "QB"].groupby("player_id").agg(
    name=("player_display_name", "first"),
    attempts=("attempts", "sum"),
    sacks=("sacks_suffered", "sum"),
    epa=("passing_epa", "sum")
).reset_index()
career["dropbacks"] = career["attempts"] + career["sacks"]
career["epa_per_db"] = career["epa"] / career["dropbacks"]

# Only QBs with real careers (2,500+ dropbacks, about 5 seasons as a starter)
career = career[career["dropbacks"] >= 2500]
career = career.merge(playoff_record, left_on="player_id", right_on="qb_id", how="left")
career[["wins", "losses"]] = career[["wins", "losses"]].fillna(0).astype(int)

display(career[["name", "dropbacks", "epa_per_db", "wins", "losses"]].sort_values("epa_per_db", ascending=False).round(3))

# Chart
fig, ax = plt.subplots(figsize=(10, 7))
others = career[career["name"] != "Dak Prescott"]
dak = career[career["name"] == "Dak Prescott"]
ax.scatter(others["epa_per_db"], others["wins"], color="#9AA5B1", s=70)
ax.scatter(dak["epa_per_db"], dak["wins"], color="#003594", s=220, edgecolor="black", zorder=3)
for _, r in career.iterrows():
    ax.annotate(r["name"].replace(" Jr.", "").split()[-1], (r["epa_per_db"], r["wins"]),
                xytext=(5, 5), textcoords="offset points", fontsize=9,
                fontweight="bold" if r["name"] == "Dak Prescott" else "normal")
ax.set_xlabel("Career EPA per dropback (regular season, 2016 to now)")
ax.set_ylabel("Playoff wins (2016 to now)")
ax.set_title("Top-8 efficiency since 2016. 2 playoff wins.", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("take1_epa_vs_playoff_wins.png", dpi=200)
plt.show()

In [ ]:
# Most recent contract for each player holds their full pay history
latest = contracts.sort_values("year_signed", ascending=False).drop_duplicates("gsis_id")

def cash_since_2016(history):
    total = 0
    if history is None:
        return 0
    for yr in history:
        if str(yr["year"]).isdigit() and 2016 <= int(yr["year"]) <= 2026 and yr["cash_paid"]:
            total += yr["cash_paid"]
    return total

latest["cash_2016_now"] = latest["season_history"].apply(cash_since_2016)

cost = career.merge(latest[["gsis_id", "cash_2016_now"]], left_on="player_id", right_on="gsis_id", how="left")
cost["cash_per_playoff_win"] = cost["cash_2016_now"] / cost["wins"].replace(0, float("nan"))

# Top 12 highest paid QBs since 2016
top_paid = cost.sort_values("cash_2016_now", ascending=False).head(12)
display(top_paid[["name", "cash_2016_now", "wins", "losses", "cash_per_playoff_win"]].round(1))

# Chart: only QBs with at least 1 playoff win
bars = top_paid.dropna(subset=["cash_per_playoff_win"]).sort_values("cash_per_playoff_win")
colors = ["#003594" if n == "Dak Prescott" else "#9AA5B1" for n in bars["name"]]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(bars["name"], bars["cash_per_playoff_win"], color=colors)
for i, v in enumerate(bars["cash_per_playoff_win"]):
    ax.text(v + 3, i, f"${v:.0f}M", va="center", fontsize=10)
ax.set_xlabel("Cash paid per playoff win since 2016 ($ millions)")
ax.set_title("Dallas has paid Dak $168M per playoff win", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("take1_cost_per_playoff_win.png", dpi=200)
plt.show()

In [ ]:
chart.to_csv("take1_qb_salary_vs_epa_2026.csv", index=False)
cost.to_csv("take1_qb_cost_per_playoff_win.csv", index=False)

## Take 2: Hire a defensive head coach
Defensive vs. offensive head coach hires since 2010, and Dallas vs. Seattle offense/defense ranks.

In [ ]:
# Regular season results with each team's head coach
sched = nfl.load_schedules().to_pandas()
reg = sched[(sched["game_type"] == "REG") & (sched["result"].notna())]

home = reg.assign(team=reg["home_team"], coach=reg["home_coach"],
                  win=(reg["result"] > 0) * 1.0 + (reg["result"] == 0) * 0.5)
away = reg.assign(team=reg["away_team"], coach=reg["away_coach"],
                  win=(reg["result"] < 0) * 1.0 + (reg["result"] == 0) * 0.5)
coach_games = pd.concat([home, away])[["season", "team", "coach", "win"]]

# First season for each coach with each team
coach_games["first_season"] = coach_games.groupby(["coach", "team"])["season"].transform("min")

# Hires from 2010 to 2024, first 3 seasons only
hires = coach_games[(coach_games["first_season"] >= 2010) & (coach_games["first_season"] <= 2024)]
hires = hires[hires["season"] < hires["first_season"] + 3]
hires = hires.groupby(["coach", "team", "first_season"]).agg(
    games=("win", "size"), wins=("win", "sum")).reset_index()
hires = hires[hires["games"] >= 10]   # drop 1-game interim stints
hires["win_pct"] = hires["wins"] / hires["games"]

# Each coach's background before becoming head coach (D = defense, O = offense, ST = special teams)
background = {
    "Leslie Frazier": "D", "Pete Carroll": "D", "John Fox": "D", "Romeo Crennel": "D", "Ron Rivera": "D",
    "Greg Schiano": "D", "Jeff Fisher": "D", "Dennis Allen": "D", "Chuck Pagano": "D", "Gus Bradley": "D",
    "Mike Zimmer": "D", "Lovie Smith": "D", "Mike Pettine": "D", "Todd Bowles": "D", "Rex Ryan": "D",
    "Jack Del Rio": "D", "Dan Quinn": "D", "Vance Joseph": "D", "Sean McDermott": "D", "Matt Patricia": "D",
    "Mike Vrabel": "D", "Steve Wilks": "D", "Vic Fangio": "D", "Brian Flores": "D", "Robert Saleh": "D",
    "Brandon Staley": "D", "Matt Eberflus": "D", "Jonathan Gannon": "D", "DeMeco Ryans": "D",
    "Jerod Mayo": "D", "Mike Macdonald": "D", "Antonio Pierce": "D", "Raheem Morris": "D",
    "Chan Gailey": "O", "Jason Garrett": "O", "Mike Shanahan": "O", "Hue Jackson": "O", "Jim Harbaugh": "O",
    "Mike Munchak": "O", "Pat Shurmur": "O", "Mike Mularkey": "O", "Joe Philbin": "O", "Chip Kelly": "O",
    "Doug Marrone": "O", "Rob Chudzinski": "O", "Mike McCoy": "O", "Marc Trestman": "O", "Andy Reid": "O",
    "Bruce Arians": "O", "Ken Whisenhunt": "O", "Bill O'Brien": "O", "Jim Caldwell": "O", "Jay Gruden": "O",
    "Gary Kubiak": "O", "Dirk Koetter": "O", "Ben McAdoo": "O", "Adam Gase": "O", "Doug Pederson": "O",
    "Kyle Shanahan": "O", "Anthony Lynn": "O", "Sean McVay": "O", "Frank Reich": "O", "Matt Nagy": "O",
    "Kliff Kingsbury": "O", "Freddie Kitchens": "O", "Zac Taylor": "O", "Matt LaFleur": "O",
    "Kevin Stefanski": "O", "Matt Rhule": "O", "Mike McCarthy": "O", "Urban Meyer": "O", "Arthur Smith": "O",
    "Dan Campbell": "O", "David Culley": "O", "Nick Sirianni": "O", "Mike McDaniel": "O",
    "Nathaniel Hackett": "O", "Brian Daboll": "O", "Josh McDaniels": "O", "Kevin O'Connell": "O",
    "Sean Payton": "O", "Shane Steichen": "O", "Brian Callahan": "O", "Dave Canales": "O",
    "Joe Judge": "ST",
}

# Interim coaches and relocated teams (same coach, new city) don't count as real hires
not_real_hires = [("Joe Vitt", "NO"), ("Tony Sparano", "OAK"), ("Bill Callahan", "WAS"),
                  ("Romeo Crennel", "HOU"), ("Raheem Morris", "ATL"), ("Rich Bisaccia", "LV"),
                  ("Steve Wilks", "CAR"), ("Antonio Pierce", "LV"), ("Jeff Fisher", "LA"), ("Jon Gruden", "LV")]
for coach, team in not_real_hires:
    hires = hires[~((hires["coach"] == coach) & (hires["team"] == team))]

hires["background"] = hires["coach"].map(background)
hires = hires[hires["background"].isin(["D", "O"])]

summary = hires.groupby("background").agg(
    hires=("coach", "size"),
    avg_win_pct=("win_pct", "mean"),
    winning_records=("win_pct", lambda x: (x > 0.5).mean())
).round(3)
summary.index = summary.index.map({"D": "Defensive", "O": "Offensive"})
display(summary)
display(hires.sort_values("win_pct", ascending=False).round(3))

In [ ]:
# Team points scored and allowed per game, ranked each season (1 = best)
h = reg.assign(team=reg["home_team"], pf=reg["home_score"], pa=reg["away_score"],
               win=(reg["result"] > 0) + 0.5 * (reg["result"] == 0))
a = reg.assign(team=reg["away_team"], pf=reg["away_score"], pa=reg["home_score"],
               win=(reg["result"] < 0) + 0.5 * (reg["result"] == 0))
teams = pd.concat([h, a]).groupby(["season", "team"]).agg(
    games=("pf", "size"), wins=("win", "sum"), pts_for=("pf", "mean"), pts_against=("pa", "mean")).reset_index()
teams["offense_rank"] = teams.groupby("season")["pts_for"].rank(ascending=False, method="min").astype(int)
teams["defense_rank"] = teams.groupby("season")["pts_against"].rank(ascending=True, method="min").astype(int)

dal = teams[(teams["team"] == "DAL") & (teams["season"] >= 2020)]
sea = teams[(teams["team"] == "SEA") & (teams["season"] >= 2020)]
display(dal.round(1))
display(sea.round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, df, title, hire_year, hire_label in [
    (axes[0], dal, "Dallas: the offense isn't the problem", 2025, "Schottenheimer\nhired"),
    (axes[1], sea, "Seattle: new defensive HC, new identity", 2024, "Macdonald\nhired")]:
    ax.plot(df["season"], df["offense_rank"], marker="o", color="#9AA5B1", linewidth=2, label="Offense rank (points scored)")
    ax.plot(df["season"], df["defense_rank"], marker="o", color="#003594", linewidth=3, label="Defense rank (points allowed)")
    ax.axvline(hire_year - 0.5, color="black", linestyle="--", linewidth=1)
    ax.text(hire_year - 0.45, 31, hire_label, fontsize=9)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Season")
    ax.set_xticks(df["season"])
# Dan Quinn era on the Dallas chart
axes[0].axvspan(2020.5, 2023.5, color="#003594", alpha=0.08)
axes[0].text(2021.1, 13, "Dan Quinn\nrunning the defense", fontsize=9, color="#003594")
axes[0].axvline(2023.5, color="#003594", linestyle=":", linewidth=1.5)
axes[0].text(2023.45, 24, "Quinn leaves →", fontsize=9, color="#003594", ha="right")
axes[0].set_ylabel("League rank (1 = best)")
axes[0].invert_yaxis()
axes[0].legend(loc="lower left", fontsize=9)
plt.tight_layout()
plt.savefig("take2_dallas_vs_seattle.png", dpi=200)
plt.show()

teams.to_csv("take2_team_ranks_by_season.csv", index=False)
hires.to_csv("take2_coach_hires_since_2010.csv", index=False)

## Take 3: Keep CeeDee, trade Pickens
What star WR trades returned, and CeeDee vs. Pickens production.

In [ ]:
# Cell 12: Take 3 - Star WR trade returns + CeeDee vs. Pickens
import nflreadpy as nfl
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Part A: recent star WR trades and the picks they returned ----------
trades = nfl.load_trades().to_pandas()
comps = [("Tyreek Hill", 2022, 28), ("Davante Adams", 2022, 29), ("A.J. Brown", 2022, 24),
         ("Stefon Diggs", 2020, 26), ("Jaylen Waddle", 2026, 27), ("Stefon Diggs", 2024, 30)]

rows = []
for name, season, age in comps:
    player_row = trades[(trades["pfr_name"] == name) & (trades["season"] == season)].iloc[0]
    deal = trades[(trades["trade_id"] == player_row["trade_id"]) & (trades["received"] == player_row["gave"])]
    picks = deal[deal["pick_round"].notna()].sort_values(["pick_season", "pick_round"])
    ordinal = {1: "1st", 2: "2nd", 3: "3rd", 4: "4th", 5: "5th", 6: "6th", 7: "7th"}
    counts = picks["pick_round"].astype(int).value_counts().sort_index()
    pick_text = " + ".join((f"two {ordinal[rd]}s" if n == 2 else ordinal[rd]) for rd, n in counts.items())
    rows.append({"WR": name, "Year": season, "Age": age,
                 "Traded to": player_row["received"], "Picks returned": pick_text})
wr_trades = pd.DataFrame(rows)
display(wr_trades)

# Table image for the slide
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.axis("off")
tbl = ax.table(cellText=wr_trades.values, colLabels=wr_trades.columns, loc="center", cellLoc="left",
               colWidths=[0.2, 0.1, 0.08, 0.14, 0.38])
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.6)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#003594")
        cell.set_text_props(color="white", fontweight="bold")
    elif wr_trades.iloc[r - 1]["Age"] >= 30:
        cell.set_facecolor("#F1F1F1")
ax.set_title("Star WRs in their 20s return a 1st-round pick. At 30, just a 2nd.", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("take3_wr_trade_returns.png", dpi=200, bbox_inches="tight")
plt.show()

# ---------- Part B: CeeDee vs. Pickens yards per game, 2022 to now ----------
history = nfl.load_player_stats(list(range(2022, 2027)), summary_level="reg").to_pandas()
stats26 = history[history["season"] == 2026]
duo = history[history["player_display_name"].isin(["CeeDee Lamb", "George Pickens"])].copy()
duo["yards_per_game"] = duo["receiving_yards"] / duo["games"]
display(duo[["player_display_name", "season", "recent_team", "games", "receiving_yards", "receiving_tds", "yards_per_game"]].round(1))

fig, ax = plt.subplots(figsize=(10, 5))
for name, color in [("CeeDee Lamb", "#003594"), ("George Pickens", "#9AA5B1")]:
    d = duo[duo["player_display_name"] == name].sort_values("season")
    ax.plot(d["season"], d["yards_per_game"], marker="o", linewidth=3, color=color, label=name)
    for _, r in d.iterrows():
        ax.annotate(f"{r['yards_per_game']:.0f}", (r["season"], r["yards_per_game"]),
                    xytext=(0, 8 if name == "CeeDee Lamb" else -16), textcoords="offset points", ha="center", fontsize=9, color=color)
ax.axvline(2024.5, color="black", linestyle="--", linewidth=1)
ax.text(2024.55, 50, "Pickens traded\nto Dallas", fontsize=9)
ax.set_xticks(range(2022, 2027))
ax.set_ylim(40, 115)
ax.set_xlabel("Season (2026 so far)")
ax.set_ylabel("Receiving yards per game")
ax.legend(loc="upper left")
ax.set_title("CeeDee is the better WR. That's why he's the one with trade value.", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("take3_ceedee_vs_pickens_production.png", dpi=200)
plt.show()

# ---------- Part C: CeeDee vs. Pickens side-by-side ----------
players = nfl.load_players().to_pandas()
stats25 = history[history["season"] == 2025]

def wr_line(name):
    pid = stats26.loc[stats26["player_display_name"] == name, "player_id"].iloc[0]
    birth = pd.to_datetime(players.loc[players["gsis_id"] == pid, "birth_date"].iloc[0])
    age = (pd.Timestamp.today() - birth).days / 365.25
    s25 = stats25[stats25["player_id"] == pid].iloc[0]
    s26 = stats26[stats26["player_id"] == pid].iloc[0]
    return age, s25["receiving_yards"], s26["receiving_yards"] / s26["games"]

c_age, c_25, c_ypg = wr_line("CeeDee Lamb")
p_age, p_25, p_ypg = wr_line("George Pickens")

compare = pd.DataFrame({
    "": ["Age", "2025 receiving yards", "2026 yards per game", "Salary per year",
         "Projected trade return", "If Dallas waits"],
    "CeeDee Lamb": [f"{int(c_age)}", f"{c_25:,.0f}", f"{c_ypg:.0f}", "\\$34M (signed through 2028)",
                    "1st + Day 2 pick", "Value drops every year (Diggs at 30: one 2nd)"],
    "George Pickens": [f"{int(p_age)}", f"{p_25:,.0f}", f"{p_ypg:.0f}", "\\$27.3M tag now, ~\\$31-35M on new deal",
                       "Day 2 pick at best (his last trade: 3rd + 5th)", "Can leave in free agency"],
})
display(compare)

fig, ax = plt.subplots(figsize=(12, 3.8))
ax.axis("off")
tbl = ax.table(cellText=compare.values, colLabels=compare.columns, loc="center", cellLoc="left",
               colWidths=[0.2, 0.38, 0.42])
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.7)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#003594")
        cell.set_text_props(color="white", fontweight="bold")
    elif c == 0:
        cell.set_text_props(fontweight="bold")
ax.set_title("Same price, younger player: trade CeeDee, keep Pickens", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("take3_ceedee_vs_pickens.png", dpi=200, bbox_inches="tight")
plt.show()
compare.apply(lambda col: col.str.replace("\\$", "$", regex=False)).to_csv("take3_ceedee_vs_pickens.csv", index=False)

# Save data for Power BI
wr_trades.to_csv("take3_star_wr_trades.csv", index=False)
duo.to_csv("take3_ceedee_vs_pickens_by_season.csv", index=False)

In [ ]:
import nflreadpy as nfl, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
NAVY="#003594"
def table_img(df, title, path, widths, hl=None):
    fig, ax = plt.subplots(figsize=(12, 0.55*len(df)+1.6)); ax.axis("off")
    tb = ax.table(cellText=df.values, colLabels=df.columns, loc="center", cellLoc="left", colWidths=widths)
    tb.auto_set_font_size(False); tb.set_fontsize(10.5); tb.scale(1, 1.8)
    for (r, c), cell in tb.get_celld().items():
        if r == 0: cell.set_facecolor(NAVY); cell.set_text_props(color="white", fontweight="bold")
        elif hl is not None and r-1 in hl: cell.set_facecolor("#EAF0F8"); cell.set_text_props(fontweight="bold")
        elif c == 0: cell.set_text_props(fontweight="bold")
    ax.set_title(title, fontsize=13.5, fontweight="bold")
    plt.tight_layout(); plt.savefig(path, dpi=200, bbox_inches="tight"); plt.close()

comps = pd.DataFrame([
 ["D.K. Metcalf", "2025", "SEA → PIT", "2nd (#52) + pick swap", "Seattle won the Super Bowl that season"],
 ["Yannick Ngakoue", "2020", "JAX → MIN", "2nd + 5th", "Franchise-tagged, then traded"],
 ["Dee Ford", "2019", "KC → SF", "2nd", "Franchise-tagged, then traded"],
 ["Stefon Diggs", "2024", "BUF → HOU", "2nd", "Buffalo moved on from its WR1"],
], columns=["Player", "Year", "Trade", "Return", "Note"])
table_img(comps, "Good players about to get paid still return a 2nd-round pick", "t3_comps.png", [0.18,0.08,0.14,0.24,0.36], hl=[0])

h = nfl.load_player_stats([2025, 2026], summary_level="reg").to_pandas()
pl = nfl.load_players().to_pandas()
def line(n):
    d = h[h.player_display_name == n]; pid = d.player_id.iloc[0]
    age = int((pd.Timestamp.today() - pd.to_datetime(pl.loc[pl.gsis_id == pid, "birth_date"].iloc[0])).days / 365.25)
    out = [str(age)]
    for yr in [2025, 2026]:
        r = d[d.season == yr].iloc[0]
        out.append(f"{int(r.receiving_yards):,} yds in {int(r.games)} games ({r.receiving_yards / r.games:.0f}/game)")
    return out
cmp = pd.DataFrame({"": ["Age", "2025", "2026 so far", "Contract", "Trade value", "Plan"],
  "CeeDee Lamb": line("CeeDee Lamb") + ["Already signed: \\$34M/yr through 2028", "1st + Day 2 pick", "KEEP. Still the WR1, and he wants to be here"],
  "George Pickens": line("George Pickens") + ["\\$27.3M tag now, needs ~\\$31-35M/yr next", "2nd + late pick (Ngakoue: 2nd + 5th)", "TRADE. A WR1 for someone else"]})
table_img(cmp, "Keep the better WR. Trade the one about to get paid.", "take3_ceedee_vs_pickens.png", [0.16,0.42,0.42])